In [1]:
import pandas as pd
import json 
import io
import zstandard as zstd
from tqdm import tqdm


In [2]:
import os
filepath = "./reddit/subreddits/subreddits_2025-01.zst"
print(os.path.getsize(filepath) / 1e6, "MB")

2240.562367 MB


In [3]:
filepath = "./reddit/subreddits/subreddits_meta_only_2025-01.zst"
print(os.path.getsize(filepath) / 1e6, "MB")

32.967187 MB


### Function for loading zst files

In [10]:

#Bruger chunks så den ikke crasher pga memory overload
def load_zst_json(filepath, chunk_size=10000):
    chunks = []
    buffer = []


    with open(filepath, "rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            text_stream = io.TextIOWrapper(reader, encoding="utf-8")

            num_dataframes=0
            for i, line in tqdm(enumerate(text_stream), desc= "Amount of subreddits processed"):
                buffer.append(json.loads(line)) #Converts each json line to python dictionary

                if len(buffer) >= chunk_size:
                    chunks.append(pd.DataFrame(buffer))
                    buffer = []
                    #Tilføj denne linje hvis du kun vil have et subset ud. 100 svarer til 100000 subreddits
                    #if len(chunks) >= 100: return pd.concat(chunks, ignore_index=True)

                    # gemmer fil for hver million subreddits
                    if len(chunks) >= 100:
                        df=pd.concat(chunks,ignore_index=True)
                        df.to_csv(f"./data/subreddits/dataframe_{num_dataframes}.csv", index=False)
                        # reset chunks and buffer
                        chunks = []
                        buffer = []
                        num_dataframes+=1

            # remaining rows
            if buffer:
                chunks.append(pd.DataFrame(buffer))
                
    if len(chunks) > 0:
        df = pd.concat(chunks, ignore_index=True)
        df.to_csv(f"./data/subreddits/dataframe_{num_dataframes}")

### Loading four different dataframes

In [60]:

meta_df = load_zst_json("./reddit/subreddits/subreddits_meta_only_2025-01.zst");
wiki_df = load_zst_json("./reddit/subreddits/subreddit_wikis_2025-01.zst");
rules_df = load_zst_json("./reddit/subreddits/subreddit_rules_2025-01.zst");



In [12]:
load_zst_json("./reddit/subreddits/subreddits_2025-01.zst")

Amount of subreddits processed: 21865531it [44:46, 8140.26it/s] 


### Custom display options to enable us to see all columns of subreddits_2025_01.zst

In [39]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 50)

### Display subreddits_2025_01.zst

In [66]:
subreddit_df.head()

,_meta,accept_followers,accounts_active,accounts_active_is_fuzzed,active_user_count,advertiser_category,all_original_content,allow_discovery,allow_galleries,allow_images,allow_polls,allow_prediction_contributors,allow_predictions,allow_predictions_tournament,allow_talks,allow_videogifs,allow_videos,allowed_media_in_comments,banner_background_color,banner_background_image,banner_img,banner_size,can_assign_link_flair,can_assign_user_flair,collapse_deleted_comments,comment_contribution_settings,comment_score_hide_mins,community_icon,community_reviewed,created,created_utc,description,disable_contributor_requests,display_name,display_name_prefixed,emojis_custom_size,emojis_enabled,free_form_reports,has_menu_widget,header_img,header_size,header_title,hide_ads,icon_img,icon_size,id,is_crosspostable_subreddit,is_enrolled_in_new_modmail,key_color,lang,link_flair_enabled,link_flair_position,mobile_banner_image,name,notification_level,original_content_tag_enabled,over18,prediction_leaderboard_entry_type,primary_color,public_description,public_traffic,quarantine,restrict_commenting,restrict_posting,retrieved_on,should_archive_posts,should_show_media_in_comments_setting,show_media,show_media_preview,spoilers_enabled,submission_type,submit_link_label,submit_text,submit_text_html,submit_text_label,subreddit_type,subscribers,suggested_comment_sort,title,url,user_can_flair_in_sr,user_flair_background_color,user_flair_css_class,user_flair_enabled_in_sr,user_flair_position,user_flair_richtext,user_flair_template_id,user_flair_text,user_flair_text_color,user_flair_type,user_has_favorited,user_is_banned,user_is_contributor,user_is_moderator,user_is_muted,user_is_subscriber,user_sr_flair_enabled,user_sr_theme_enabled,videostream_links_count,wiki_enabled,wls,quarantine_message,quarantine_message_html,quarantine_permissions,is_default_banner,is_default_icon,interstitial_warning_message,content_category
0,"{'earliest_comment_at': 1434158967, 'earliest_...",True,None,False,None,,False,True,True,True,True,False,False,False,False,True,True,[],,,,None,False,False,False,{},0.0,,False,1410893311,1410893311,,False,00000,r/00000,None,False,True,False,None,None,,False,,None,33l9l,True,None,,en,False,,,t5_33l9l,None,False,False,1.0,,,False,False,False,True,1739561915,False,True,False,True,True,any,,,None,,restricted,14.0,None,00000,/r/00000/,None,None,None,False,right,[],None,None,None,text,False,False,False,False,False,False,None,True,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"{'earliest_comment_at': None, 'earliest_post_a...",True,None,False,None,,False,True,True,True,True,False,False,False,False,True,True,"[giphy, static, animated, expression]",,,,None,False,False,True,"{'allowed_media_types': ['giphy', 'static', 'a...",0.0,,False,1706444476,1706444476,,False,000000,r/000000,None,False,False,False,None,None,,False,,None,aoez2i,True,None,,en,False,,,t5_aoez2i,None,False,False,1.0,,,False,False,False,True,1739603362,False,True,True,True,True,any,,,None,,restricted,1.0,None,000000,/r/000000/,None,None,None,False,right,[],None,None,None,text,False,False,False,False,False,False,None,True,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"{'earliest_comment_at': None, 'earliest_post_a...",None,None,None,None,None,None,None,None,None,None,False,False,False,False,True,True,[],None,,None,None,False,False,None,{},NaN,,None,1481764006,1481764006,None,None,00000000,r/00000000,None,False,None,False,None,None,None,None,None,None,3i0gl,False,None,None,None,None,None,None,t5_3i0gl,None,None,None,NaN,None,,None,None,None,None,1739583892,None,True,None,None,None,None,None,None,None,None,restricted,NaN,None,011,/r/00000000/,None,None,None,None,None,[],None,None,None,text,None,None,None,None,None,None,None,None,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"{'earliest_comment_at': 1495585546, 'earliest_...",None,None,None,None,None,None,None,None,None,None,False,False,False,False,True,False,"[giphy, static, animated, expression]",None,,None,None,False,False,None,"{'allowed_media_types':

In [65]:
description =subreddit_df["description"]
filtered_description = description[(description.notna()) & (description.str.strip() != "")]
filtered_description

15        if there was a list of all the subreddits, lis...
18                                               no sidebar
23        this here is a safe zone for cross-posts that ...
26                                                      000
32                                                    hello
                                ...                        
999941                                    piatti da provare
999942         il sito di ecommerce per la casa e la cucina
999945    Qui si celebrano i disastri culinari, le torte...
999951    Benvenuti!\nQuesto è il subreddit italiano per...
999987    Photos and videos related to Cucks cumming, wi...
Name: description, Length: 190264, dtype: object

In [58]:
subreddit_df.loc[subreddit_df["display_name"] == "leagueoflegends"]

,_meta,accept_followers,accounts_active,accounts_active_is_fuzzed,active_user_count,advertiser_category,all_original_content,allow_discovery,allow_galleries,allow_images,allow_polls,allow_prediction_contributors,allow_predictions,allow_predictions_tournament,allow_talks,allow_videogifs,allow_videos,allowed_media_in_comments,banner_background_color,banner_background_image,banner_img,banner_size,can_assign_link_flair,can_assign_user_flair,collapse_deleted_comments,comment_contribution_settings,comment_score_hide_mins,community_icon,community_reviewed,created,created_utc,description,disable_contributor_requests,display_name,display_name_prefixed,emojis_custom_size,emojis_enabled,free_form_reports,has_menu_widget,header_img,header_size,header_title,hide_ads,icon_img,icon_size,id,is_crosspostable_subreddit,is_enrolled_in_new_modmail,key_color,lang,link_flair_enabled,link_flair_position,mobile_banner_image,name,notification_level,original_content_tag_enabled,over18,prediction_leaderboard_entry_type,primary_color,public_description,public_traffic,quarantine,restrict_commenting,restrict_posting,retrieved_on,should_archive_posts,should_show_media_in_comments_setting,show_media,show_media_preview,spoilers_enabled,submission_type,submit_link_label,submit_text,submit_text_html,submit_text_label,subreddit_type,subscribers,suggested_comment_sort,title,url,user_can_flair_in_sr,user_flair_background_color,user_flair_css_class,user_flair_enabled_in_sr,user_flair_position,user_flair_richtext,user_flair_template_id,user_flair_text,user_flair_text_color,user_flair_type,user_has_favorited,user_is_banned,user_is_contributor,user_is_moderator,user_is_muted,user_is_subscriber,user_sr_flair_enabled,user_sr_theme_enabled,videostream_links_count,wiki_enabled,wls,quarantine_message,quarantine_message_html,quarantine_permissions


### Display meta subreddit

In [11]:
print(rules_df.shape)
print(rules_df.columns.tolist())
print(rules_df["_meta"][2131])
rules_df.head()

(1638637, 3)
['_meta', 'display_name', 'id']
{'earliest_comment': None, 'earliest_post': 1665033847, 'num_comments': None, 'num_comments_updated_at': None, 'num_posts': 2, 'num_posts_updated_at': 1739731892}


,_meta,display_name,id
0,"{'earliest_comment': None, 'earliest_post': 15...",*cohold00009,7h69l
1,"{'earliest_comment': None, 'earliest_post': 15...",*cohold00022,7hzhb
2,"{'earliest_comment': None, 'earliest_post': 15...",*polhold00214,edn2z
3,"{'earliest_comment': None, 'earliest_post': 15...",*polhold00575,7hje8
4,"{'earliest_comment': None, 'earliest_post': 16...",*polhold01359,4ly4qt


### Display meta subreddit

In [2]:
meta_df.head()

,_meta,display_name,id
0,"{'earliest_comment': None, 'earliest_post': 15...",*cohold00009,7h69l
1,"{'earliest_comment': None, 'earliest_post': 15...",*cohold00022,7hzhb
2,"{'earliest_comment': None, 'earliest_post': 15...",*polhold00214,edn2z
3,"{'earliest_comment': None, 'earliest_post': 15...",*polhold00575,7hje8
4,"{'earliest_comment': None, 'earliest_post': 16...",*polhold01359,4ly4qt


In [3]:
print(meta_df.shape)
print(meta_df.columns.tolist())
print(meta_df["_meta"][2131])

(1638637, 3)
['_meta', 'display_name', 'id']
{'earliest_comment': None, 'earliest_post': 1665033847, 'num_comments': None, 'num_comments_updated_at': None, 'num_posts': 2, 'num_posts_updated_at': 1739731892}


### Display wiki subreddit

In [4]:
wiki_df.head()

,content,path,retrieved_on,revision_author,revision_author_id,revision_date,revision_reason
0,Here are possible solutions you can try that I...,/r/001Guy001/wiki/index,1737155927,001Guy001,1299b0,2024-11-17T17:45:02+00:00,NaN
1,:::\n\n“Welcome to the most ancient conspiracy...,/r/00AG9603/wiki/index,1737235480,minimalista,7km5s,2024-09-12T23:35:46+00:00,NaN
2,\nIndividuals are encouraged and controlled in...,/r/00AG9603/wiki/index/open_login_-_reality_gl...,1737306258,minimalista,7km5s,2020-01-16T13:00:08+00:00,NaN
3,Other Pages: \r\n[People](https://reddit.com/...,/r/00weareready00/wiki/index,1737155927,MegaMCAlly,ayu74,2016-06-22T19:36:59+00:00,NaN
4,[Back to Main Page](https://www.reddit.com/r/0...,/r/00weareready00/wiki/people,1737155925,MegaMCAlly,ayu74,2016-06-22T23:53:01+00:00,NaN


In [5]:
print(wiki_df.shape)
print(wiki_df.columns.tolist())
print(wiki_df["content"][4])


(322964, 7)
['content', 'path', 'retrieved_on', 'revision_author', 'revision_author_id', 'revision_date', 'revision_reason']
[Back to Main Page](https://www.reddit.com/r/00weareready00/wiki/index)

A list of people known to be involved in the ARG.

Known to be Official:

        Name: 
            Nigel
        Stats: 
            Protagonist, Main Character
        Twitter: 
            @00weareready00

        Name: 
            Watch
        Stats: 
            Antagonist(s?)
        Twitter: 
            @wa_tch_

        Name: 
            Matt
        Stats: 
            Protagonist (Exact role undetermined)
        Twitter: 
            @Dont1worry

Unsure if Official or Not (as of now):

    N/A

Skitzes (Gamejackers):

        Name: 
            ???
        Stats: 
            Skitz
        Twitter: 
            @15xanswersx15


### Test for subreddit

In [6]:
def get_subreddit_name(path):
    subreddit_name = path.split("/")[2]
    return subreddit_name

wiki_df["SUBREDDIT_NAME"] = wiki_df["path"].apply(get_subreddit_name)
selected_wiki_df = wiki_df[["SUBREDDIT_NAME", "content"]]  

In [7]:
selected_wiki_df.to_csv("./data/wiki_df.csv", index=False)

In [8]:
selected_wiki_df.isin(["soccer"]).any()

SUBREDDIT_NAME     True
content           False
dtype: bool

In [8]:
import os
size = os.path.getsize("./data/wiki_df.csv")
print(f"File size: {size} bytes")


File size: 1747051129 bytes
